# 07a — GPT-OSS 20B Baseline Evaluation (Temporary)

Standalone baseline evaluation of `gpt-oss:20b` on the same ground-truth
dataset used in the main pipeline. This notebook is **self-contained** and
can be deleted without affecting anything else.

Forces full GPU offload to RTX 5090.

In [ ]:
# =============================================================================
# Setup — Imports, configuration, data loading
# =============================================================================
import os, re, time
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import ollama
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

# ─── Paths ───
notebook_dir = Path.cwd()
project_root = notebook_dir if (notebook_dir / 'Data').exists() else notebook_dir.parent
DATA_DIR     = project_root / 'Data'
RESULTS_DIR  = DATA_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

# ─── Model ───
MODEL_NAME = 'gpt-oss:20b'
SAVE_NAME  = 'gpt_oss_20b'
TEMPERATURE = 0.2
RANDOM_STATE = 42

# RTX 5090: offload ALL layers to GPU (999 = all layers)
NUM_GPU_LAYERS = 999

# ─── Load ground truth (same logic as main notebook) ───
ground_truth = pd.read_csv(DATA_DIR / 'ground_truth_validation_dataset.csv')

review_labels = (ground_truth.groupby('review_doi')['label']
                 .agg(['sum', 'count']).reset_index())
review_labels.columns = ['review_doi', 'n_included', 'n_total']
review_labels['n_excluded'] = review_labels['n_total'] - review_labels['n_included']
reviews_with_both = review_labels[
    (review_labels['n_included'] > 0) & (review_labels['n_excluded'] > 0)
]
eval_data = ground_truth[
    ground_truth['review_doi'].isin(reviews_with_both['review_doi'])
].copy()

eval_sample = eval_data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# ─── Verify model is available ───
available = [m.model for m in ollama.list().models]
if not any(MODEL_NAME in a for a in available):
    raise RuntimeError(f'{MODEL_NAME} not found! Run: ollama pull {MODEL_NAME}')

print(f'Model:     {MODEL_NAME}')
print(f'GPU layers: {NUM_GPU_LAYERS} (full offload to RTX 5090)')
print(f'Eval set:  {len(eval_sample):,}  '
      f'({(eval_sample["label"]==1).sum()} INC / {(eval_sample["label"]==0).sum()} EXC)')
print(f'Temperature: {TEMPERATURE}')

Model:     gpt-oss:20b
GPU layers: 999 (full offload to RTX 5090)
Eval set:  5,721  (899 INC / 4822 EXC)
Temperature: 0.2


In [ ]:
# =============================================================================
# Core functions
# =============================================================================

ZERO_SHOT_PROMPT = (
    "You are assisting with title/abstract screening for a Cochrane systematic review.\n\n"
    "Given the review's scope (title and abstract) and a candidate paper's title "
    "and abstract, decide whether the paper is relevant to the review.\n\n"
    "=== REVIEW ===\n"
    "Title: {review_title}\n\n"
    "Abstract: {review_abstract}\n\n"
    "=== CANDIDATE PAPER ===\n"
    "Title: {paper_title}\n\n"
    "Abstract: {paper_abstract}\n\n"
    "=== DECISION ===\n"
    "Based on the information above, should this paper be included in the review?\n\n"
    "Respond with exactly one word: INCLUDE or EXCLUDE"
)


def create_prompt(row):
    return ZERO_SHOT_PROMPT.format(
        review_title    = str(row['review_title'])[:500],
        review_abstract = str(row['review_abstract'])[:2000],
        paper_title     = str(row['paper_title'])[:300],
        paper_abstract  = str(row['paper_abstract'])[:2000],
    )


def extract_decision(response: str) -> int:
    if not response or not response.strip():
        return -1
    up = response.upper()
    m = re.search(r'DECISION:\s*(INCLUDE|EXCLUDE)', up)
    if m:
        return 1 if m.group(1) == 'INCLUDE' else 0
    for line in reversed(up.strip().split('\n')[-5:]):
        line = line.strip()
        if line == 'INCLUDE': return 1
        if line == 'EXCLUDE': return 0
    inc, exc = up.rfind('INCLUDE'), up.rfind('EXCLUDE')
    if inc > exc: return 1
    if exc > inc: return 0
    return -1


def llm_call(prompt):
    t0 = time.time()
    resp = ollama.generate(
        model=MODEL_NAME,
        prompt=prompt,
        options={
            'temperature': TEMPERATURE,
            'num_predict': 2048,
            'num_gpu': NUM_GPU_LAYERS,   # force full GPU offload
        },
    )
    return resp.get('response', ''), time.time() - t0


def score(labels, preds):
    mask = [p != -1 for p in preds]
    yt = [l for l, m in zip(labels, mask) if m]
    yp = [p for p, m in zip(preds, mask) if m]
    n = sum(mask)
    if n == 0:
        return dict(valid=0, acc=0, prec=0, rec=0, f1=0)
    return dict(
        valid=n,
        acc  = round(accuracy_score(yt, yp), 4),
        prec = round(precision_score(yt, yp, zero_division=0), 4),
        rec  = round(recall_score(yt, yp, zero_division=0), 4),
        f1   = round(f1_score(yt, yp, zero_division=0), 4),
    )

print('Functions defined.')

Functions defined.


In [ ]:
# =============================================================================
# Run baseline evaluation — gpt-oss:120b zero-shot
# =============================================================================
print(f"{'='*60}")
print(f'Evaluating {MODEL_NAME}  (GPU: RTX 5090, full offload)')
print(f"{'='*60}")

preds, resps, times = [], [], []
for _, row in tqdm(eval_sample.iterrows(), total=len(eval_sample),
                   desc=MODEL_NAME):
    prompt = create_prompt(row)
    try:
        resp, elapsed = llm_call(prompt)
        pred = extract_decision(resp)
    except Exception as e:
        resp, pred, elapsed = str(e), -1, 0
    preds.append(pred)
    resps.append(resp)
    times.append(elapsed)

# Build & save results
results_df = eval_sample[['review_doi', 'paper_title', 'label']].copy()
results_df['prediction'] = preds
results_df['response']   = resps
results_df['time_sec']   = [round(t, 2) for t in times]

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = RESULTS_DIR / f'eval_{SAVE_NAME}_zero_shot_{ts}.csv'
results_df.to_csv(out_path, index=False)
print(f'\n\u2713 Saved to {out_path.name}')

Evaluating gpt-oss:20b  (GPU: RTX 5090, full offload)


gpt-oss:20b:   0%|          | 0/5721 [00:00<?, ?it/s]


✓ Saved to eval_gpt_oss_20b_zero_shot_20260307_012857.csv


In [ ]:
# =============================================================================
# Results
# =============================================================================
labels = eval_sample['label'].tolist()
metrics = score(labels, preds)
metrics['time_min'] = round(sum(times) / 60, 1)

print(f"\n{'='*60}")
print(f'{MODEL_NAME}  RESULTS  ({metrics["valid"]}/{len(labels)} valid)')
print(f"{'='*60}")
print(f'  Acc  = {metrics["acc"]:.4f}')
print(f'  Prec = {metrics["prec"]:.4f}')
print(f'  Rec  = {metrics["rec"]:.4f}')
print(f'  F1   = {metrics["f1"]:.4f}')
print(f'  Time = {metrics["time_min"]:.1f} min')

# Confusion matrix
vt = [l for l, p in zip(labels, preds) if p != -1]
vp = [p for p in preds if p != -1]
if len(vt) > 0:
    cm = confusion_matrix(vt, vp)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        print(f'\n  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
        print(f'  FNR = {fn/(tp+fn):.1%}  |  FPR = {fp/(fp+tn):.1%}')

print(f"\n{'='*60}")
print('Done. This notebook can be safely deleted.')


gpt-oss:20b  RESULTS  (5721/5721 valid)
  Acc  = 0.8748
  Prec = 0.9067
  Rec  = 0.2269
  F1   = 0.3630
  Time = 218.1 min

  TP=204  FP=21  FN=695  TN=4801
  FNR = 77.3%  |  FPR = 0.4%

Done. This notebook can be safely deleted.


---
## mistral-small3.2:24b Baseline Evaluation

In [ ]:
# =============================================================================
# Run baseline evaluation — mistral-small3.2:24b zero-shot
# =============================================================================
MODEL_NAME_2 = 'mistral-small3.2:24b'
SAVE_NAME_2  = 'mistral_small3.2_24b'

# Verify model is available
available = [m.model for m in ollama.list().models]
if not any(MODEL_NAME_2 in a for a in available):
    raise RuntimeError(f'{MODEL_NAME_2} not found! Run: ollama pull {MODEL_NAME_2}')

print(f"{'='*60}")
print(f'Evaluating {MODEL_NAME_2}  (GPU: RTX 5090, full offload)')
print(f"{'='*60}")

preds_2, resps_2, times_2 = [], [], []
for _, row in tqdm(eval_sample.iterrows(), total=len(eval_sample),
                   desc=MODEL_NAME_2):
    prompt = create_prompt(row)
    try:
        t0 = time.time()
        resp = ollama.generate(
            model=MODEL_NAME_2,
            prompt=prompt,
            options={
                'temperature': TEMPERATURE,
                'num_predict': 2048,
                'num_gpu': NUM_GPU_LAYERS,
            },
        )
        resp_text = resp.get('response', '')
        elapsed = time.time() - t0
        pred = extract_decision(resp_text)
    except Exception as e:
        resp_text, pred, elapsed = str(e), -1, 0
    preds_2.append(pred)
    resps_2.append(resp_text)
    times_2.append(elapsed)

# Build & save results
results_df_2 = eval_sample[['review_doi', 'paper_title', 'label']].copy()
results_df_2['prediction'] = preds_2
results_df_2['response']   = resps_2
results_df_2['time_sec']   = [round(t, 2) for t in times_2]

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path_2 = RESULTS_DIR / f'eval_{SAVE_NAME_2}_zero_shot_{ts}.csv'
results_df_2.to_csv(out_path_2, index=False)
print(f'\n\u2713 Saved to {out_path_2.name}')

Evaluating mistral-small3.2:24b  (GPU: RTX 5090, full offload)


mistral-small3.2:24b:   0%|          | 0/5721 [00:00<?, ?it/s]


✓ Saved to eval_mistral_small3.2_24b_zero_shot_20260307_024451.csv


In [ ]:
# =============================================================================
# mistral-small3.2:24b — Results
# =============================================================================
labels_2 = eval_sample['label'].tolist()
metrics_2 = score(labels_2, preds_2)
metrics_2['time_min'] = round(sum(times_2) / 60, 1)

print(f"\n{'='*60}")
print(f'{MODEL_NAME_2}  RESULTS  ({metrics_2["valid"]}/{len(labels_2)} valid)')
print(f"{'='*60}")
print(f'  Acc  = {metrics_2["acc"]:.4f}')
print(f'  Prec = {metrics_2["prec"]:.4f}')
print(f'  Rec  = {metrics_2["rec"]:.4f}')
print(f'  F1   = {metrics_2["f1"]:.4f}')
print(f'  Time = {metrics_2["time_min"]:.1f} min')

# Confusion matrix
vt_2 = [l for l, p in zip(labels_2, preds_2) if p != -1]
vp_2 = [p for p in preds_2 if p != -1]
if len(vt_2) > 0:
    cm_2 = confusion_matrix(vt_2, vp_2)
    if cm_2.shape == (2, 2):
        tn2, fp2, fn2, tp2 = cm_2.ravel()
        print(f'\n  TP={tp2}  FP={fp2}  FN={fn2}  TN={tn2}')
        print(f'  FNR = {fn2/(tp2+fn2):.1%}  |  FPR = {fp2/(fp2+tn2):.1%}')

print(f"\n{'='*60}")
print('Done. This notebook can be safely deleted.')


mistral-small3.2:24b  RESULTS  (5721/5721 valid)
  Acc  = 0.9147
  Prec = 0.8716
  Rec  = 0.5362
  F1   = 0.6639
  Time = 64.0 min

  TP=482  FP=71  FN=417  TN=4751
  FNR = 46.4%  |  FPR = 1.5%

Done. This notebook can be safely deleted.


## gemma3:27b Baseline Evaluation

In [ ]:
# =============================================================================
# Run baseline evaluation — gemma3:27b zero-shot
# =============================================================================
MODEL_NAME_3 = 'gemma3:27b'
SAVE_NAME_3  = 'gemma3_27b'

# Verify model is available
available = [m.model for m in ollama.list().models]
if not any(MODEL_NAME_3 in a for a in available):
    raise RuntimeError(f'{MODEL_NAME_3} not found! Run: ollama pull {MODEL_NAME_3}')

print(f"{'='*60}")
print(f'Evaluating {MODEL_NAME_3}  (GPU: RTX 5090, full offload)')
print(f"{'='*60}")

preds_3, resps_3, times_3 = [], [], []
for _, row in tqdm(eval_sample.iterrows(), total=len(eval_sample),
                   desc=MODEL_NAME_3):
    prompt = create_prompt(row)
    try:
        t0 = time.time()
        resp = ollama.generate(
            model=MODEL_NAME_3,
            prompt=prompt,
            options={
                'temperature': TEMPERATURE,
                'num_predict': 2048,
                'num_gpu': NUM_GPU_LAYERS,
            },
        )
        resp_text = resp.get('response', '')
        elapsed = time.time() - t0
        pred = extract_decision(resp_text)
    except Exception as e:
        resp_text, pred, elapsed = str(e), -1, 0
    preds_3.append(pred)
    resps_3.append(resp_text)
    times_3.append(elapsed)

# Build & save results
results_df_3 = eval_sample[['review_doi', 'paper_title', 'label']].copy()
results_df_3['prediction'] = preds_3
results_df_3['response']   = resps_3
results_df_3['time_sec']   = [round(t, 2) for t in times_3]

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path_3 = RESULTS_DIR / f'eval_{SAVE_NAME_3}_zero_shot_{ts}.csv'
results_df_3.to_csv(out_path_3, index=False)
print(f'\n\u2713 Saved to {out_path_3.name}')

Evaluating gemma3:27b  (GPU: RTX 5090, full offload)


gemma3:27b:   0%|          | 0/5721 [00:00<?, ?it/s]


✓ Saved to eval_gemma3_27b_zero_shot_20260307_043306.csv


In [ ]:
# =============================================================================
# gemma3:27b — Results
# =============================================================================
labels_3 = eval_sample['label'].tolist()
metrics_3 = score(labels_3, preds_3)
metrics_3['time_min'] = round(sum(times_3) / 60, 1)

print(f"\n{'='*60}")
print(f'{MODEL_NAME_3}  RESULTS  ({metrics_3["valid"]}/{len(labels_3)} valid)')
print(f"{'='*60}")
print(f'  Acc  = {metrics_3["acc"]:.4f}')
print(f'  Prec = {metrics_3["prec"]:.4f}')
print(f'  Rec  = {metrics_3["rec"]:.4f}')
print(f'  F1   = {metrics_3["f1"]:.4f}')
print(f'  Time = {metrics_3["time_min"]:.1f} min')

# Confusion matrix
vt_3 = [l for l, p in zip(labels_3, preds_3) if p != -1]
vp_3 = [p for p in preds_3 if p != -1]
if len(vt_3) > 0:
    cm_3 = confusion_matrix(vt_3, vp_3)
    if cm_3.shape == (2, 2):
        tn3, fp3, fn3, tp3 = cm_3.ravel()
        print(f'\n  TP={tp3}  FP={fp3}  FN={fn3}  TN={tn3}')
        print(f'  FNR = {fn3/(tp3+fn3):.1%}  |  FPR = {fp3/(fp3+tn3):.1%}')

print(f"\n{'='*60}")
print('Done. This notebook can be safely deleted.')


gemma3:27b  RESULTS  (5721/5721 valid)
  Acc  = 0.9234
  Prec = 0.8954
  Rec  = 0.5806
  F1   = 0.7045
  Time = 96.3 min

  TP=522  FP=61  FN=377  TN=4761
  FNR = 41.9%  |  FPR = 1.3%

Done. This notebook can be safely deleted.


# Claude Opus 4.6 — Frontier Reasoning Model (via Anthropic API)

In [ ]:
# =============================================================================
# Run baseline evaluation — Claude Opus 4.6 zero-shot (Anthropic API)
# =============================================================================
import anthropic
from dotenv import load_dotenv

load_dotenv(project_root / '.env')

MODEL_NAME_4 = 'claude-opus-4-6'
SAVE_NAME_4  = 'claude_opus_4_6'

# ─── API client (reads ANTHROPIC_API_KEY from .env) ───────────────────────
client = anthropic.Anthropic()   # loaded from .env

# ─── Cost estimation ──────────────────────────────────────────────────────
# ~1,200 input tokens/sample × 5,721 samples ≈ 6.9 M input tokens → ~$34
# ~10 output tokens/sample × 5,721 samples ≈ 57 K output tokens    → ~$1.4
# Estimated total: ~$36  (at $5/$25 per MTok)
print(f"Evaluating {MODEL_NAME_4}  ({len(eval_sample):,} samples)")
print(f"Estimated API cost: ~$36 USD")
print(f"{'='*60}")

preds_4, resps_4, times_4 = [], [], []
for i, (_, row) in enumerate(tqdm(eval_sample.iterrows(), total=len(eval_sample),
                                   desc=MODEL_NAME_4)):
    prompt = create_prompt(row)
    try:
        t0 = time.time()
        message = client.messages.create(
            model=MODEL_NAME_4,
            max_tokens=256,
            temperature=TEMPERATURE,
            messages=[{"role": "user", "content": prompt}],
        )
        resp_text = message.content[0].text
        elapsed = time.time() - t0
        pred = extract_decision(resp_text)
    except anthropic.RateLimitError:
        # Back off on rate limit
        time.sleep(30)
        try:
            t0 = time.time()
            message = client.messages.create(
                model=MODEL_NAME_4,
                max_tokens=256,
                temperature=TEMPERATURE,
                messages=[{"role": "user", "content": prompt}],
            )
            resp_text = message.content[0].text
            elapsed = time.time() - t0
            pred = extract_decision(resp_text)
        except Exception as e:
            resp_text, pred, elapsed = str(e), -1, 0
    except Exception as e:
        resp_text, pred, elapsed = str(e), -1, 0

    preds_4.append(pred)
    resps_4.append(resp_text)
    times_4.append(elapsed)

# Build & save results
results_df_4 = eval_sample[['review_doi', 'paper_title', 'label']].copy()
results_df_4['prediction'] = preds_4
results_df_4['response']   = resps_4
results_df_4['time_sec']   = [round(t, 2) for t in times_4]

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path_4 = RESULTS_DIR / f'eval_{SAVE_NAME_4}_zero_shot_{ts}.csv'
results_df_4.to_csv(out_path_4, index=False)
print(f'\n✓ Saved to {out_path_4.name}')

Evaluating claude-opus-4-6  (5,721 samples)
Estimated API cost: ~$36 USD


claude-opus-4-6:   0%|          | 0/5721 [00:00<?, ?it/s]


✓ Saved to eval_claude_opus_4_6_zero_shot_20260307_103823.csv


In [ ]:
# =============================================================================
# Claude Opus 4.6 — Results
# =============================================================================
labels_4 = eval_sample['label'].tolist()
metrics_4 = score(labels_4, preds_4)
metrics_4['time_min'] = round(sum(times_4) / 60, 1)

print(f"\n{'='*60}")
print(f'{MODEL_NAME_4}  RESULTS  ({metrics_4["valid"]}/{len(labels_4)} valid)')
print(f"{'='*60}")
print(f'  Acc  = {metrics_4["acc"]:.4f}')
print(f'  Prec = {metrics_4["prec"]:.4f}')
print(f'  Rec  = {metrics_4["rec"]:.4f}')
print(f'  F1   = {metrics_4["f1"]:.4f}')
print(f'  Time = {metrics_4["time_min"]:.1f} min')

# Confusion matrix
vt_4 = [l for l, p in zip(labels_4, preds_4) if p != -1]
vp_4 = [p for p in preds_4 if p != -1]
if len(vt_4) > 0:
    cm_4 = confusion_matrix(vt_4, vp_4)
    if cm_4.shape == (2, 2):
        tn4, fp4, fn4, tp4 = cm_4.ravel()
        print(f'\n  TP={tp4}  FP={fp4}  FN={fn4}  TN={tn4}')
        print(f'  FNR = {fn4/(tp4+fn4):.1%}  |  FPR = {fp4/(fp4+tn4):.1%}')

print(f"\n{'='*60}")
print('Done.')


claude-opus-4-6  RESULTS  (5720/5721 valid)
  Acc  = 0.8832
  Prec = 0.8621
  Rec  = 0.3059
  F1   = 0.4516
  Time = 159.2 min

  TP=275  FP=44  FN=624  TN=4777
  FNR = 69.4%  |  FPR = 0.9%

Done.


In [ ]:
# =============================================================================
# Analyze Claude Opus 4.6 responses — understand the failure mode
# =============================================================================
import pandas as pd

r = pd.read_csv(RESULTS_DIR / 'eval_claude_opus_4_6_zero_shot_20260307_103823.csv')

# Look at false negatives (label=1, prediction=0) — papers Claude MISSED
fn_mask = (r['label'] == 1) & (r['prediction'] == 0)
fn_examples = r[fn_mask].head(10)

print(f"Total FN (missed includes): {fn_mask.sum()}")
print(f"Total predictions: include={r['prediction'].eq(1).sum()}, exclude={r['prediction'].eq(0).sum()}, invalid={r['prediction'].eq(-1).sum()}")
print(f"\n{'='*60}")
print("Sample FALSE NEGATIVE responses (Claude said EXCLUDE but should be INCLUDE):\n")

for _, row in fn_examples.iterrows():
    resp = str(row['response'])[:300]
    print(f"Paper: {row['paper_title'][:80]}")
    print(f"Response: {resp}")
    print(f"---")

Total FN (missed includes): 624
Total predictions: include=319, exclude=5401, invalid=1

Sample FALSE NEGATIVE responses (Claude said EXCLUDE but should be INCLUDE):

Paper: Cluster-randomized trial on complementary and responsive feeding education to ca
Response: EXCLUDE
---
Paper: A participatory and capacity-building approach to healthy eating and physical ac
Response: EXCLUDE
---
Paper: Activating and guiding the engagement of seniors with online social networking: 
Response: EXCLUDE
---
Paper: A model intervention for elder abuse and dementia
Response: EXCLUDE
---
Paper: Assessing outcomes and falls-risk reduction for homebound older adults through a
Response: EXCLUDE
---
Paper: Lessons learned and challenges in building a Filipino health coalition
Response: EXCLUDE
---
Paper: Development and exploratory cluster-randomised opportunistic trial of a theory-b
Response: EXCLUDE
---
Paper: Do we really need another meeting? Lessons from the Los Angeles County Elder Abu
Response: EXCLUD

# Claude Opus 4.6 — Optimised Prompt (sensitivity-focused + extended thinking)

In [ ]:
# =============================================================================
# Run optimised evaluation — Claude Opus 4.6 (sensitivity-focused prompt + extended thinking)
# =============================================================================
MODEL_NAME_5 = 'claude-opus-4-6'
SAVE_NAME_5  = 'claude_opus_4_6_optimised'

# ─── System prompt: establish screening philosophy ────────────────────────
SYSTEM_PROMPT_5 = (
    "You are an expert systematic review screener for Cochrane reviews. "
    "Your task is title/abstract screening — the FIRST pass in a two-stage process. "
    "Papers you INCLUDE will undergo full-text review later, so false inclusions "
    "are acceptable and expected.\n\n"
    "CRITICAL PRINCIPLE: In systematic review screening, SENSITIVITY (recall) is "
    "far more important than specificity. Missing a relevant paper is a serious "
    "error; including a borderline paper is not.\n\n"
    "WHEN IN DOUBT, INCLUDE THE PAPER.\n\n"
    "INCLUDE if the paper MIGHT be relevant — even tangentially — to the review's "
    "scope, population, intervention, or outcomes.\n"
    "EXCLUDE only if the paper is CLEARLY and OBVIOUSLY irrelevant."
)

# ─── Optimised user prompt ────────────────────────────────────────────────
OPTIMISED_PROMPT = (
    "=== REVIEW SCOPE ===\n"
    "Title: {review_title}\n\n"
    "Abstract: {review_abstract}\n\n"
    "=== CANDIDATE PAPER ===\n"
    "Title: {paper_title}\n\n"
    "Abstract: {paper_abstract}\n\n"
    "=== SCREENING DECISION ===\n"
    "Could this paper potentially be relevant to the review above? "
    "Remember: err on the side of inclusion. Only exclude if clearly irrelevant.\n\n"
    "Respond with exactly one word: INCLUDE or EXCLUDE"
)

def create_prompt_5(row):
    return OPTIMISED_PROMPT.format(
        review_title    = str(row['review_title'])[:500],
        review_abstract = str(row['review_abstract'])[:2000],
        paper_title     = str(row['paper_title'])[:300],
        paper_abstract  = str(row['paper_abstract'])[:2000],
    )

# ─── Cost estimation (with extended thinking) ────────────────────────────
# Extended thinking adds ~500-1000 thinking tokens per call
# Input: ~1,300 tokens/sample × 5,721 ≈ 7.4 M → ~$37
# Output: ~600 tokens/sample × 5,721 ≈ 3.4 M   → ~$86
# Estimated total: ~$123  (thinking tokens are billed at output rate)
print(f"Evaluating {MODEL_NAME_5} (OPTIMISED)  ({len(eval_sample):,} samples)")
print(f"Estimated API cost: ~$123 USD (extended thinking enabled)")
print(f"{'='*60}")

preds_5, resps_5, times_5 = [], [], []
for i, (_, row) in enumerate(tqdm(eval_sample.iterrows(), total=len(eval_sample),
                                   desc=f'{MODEL_NAME_5} (opt)')):
    prompt = create_prompt_5(row)
    try:
        t0 = time.time()
        message = client.messages.create(
            model=MODEL_NAME_5,
            max_tokens=16000,
            temperature=1,        # required by Anthropic when using extended thinking
            thinking={
                "type": "enabled",
                "budget_tokens": 8000,  # allow up to 8K tokens of reasoning
            },
            system=SYSTEM_PROMPT_5,
            messages=[{"role": "user", "content": prompt}],
        )
        # Extract text from response (skip thinking blocks)
        resp_text = ''.join(
            block.text for block in message.content
            if block.type == 'text'
        )
        elapsed = time.time() - t0
        pred = extract_decision(resp_text)
    except anthropic.RateLimitError:
        time.sleep(30)
        try:
            t0 = time.time()
            message = client.messages.create(
                model=MODEL_NAME_5,
                max_tokens=16000,
                temperature=1,
                thinking={
                    "type": "enabled",
                    "budget_tokens": 8000,
                },
                system=SYSTEM_PROMPT_5,
                messages=[{"role": "user", "content": prompt}],
            )
            resp_text = ''.join(
                block.text for block in message.content
                if block.type == 'text'
            )
            elapsed = time.time() - t0
            pred = extract_decision(resp_text)
        except Exception as e:
            resp_text, pred, elapsed = str(e), -1, 0
    except Exception as e:
        resp_text, pred, elapsed = str(e), -1, 0

    preds_5.append(pred)
    resps_5.append(resp_text)
    times_5.append(elapsed)

# Build & save results
results_df_5 = eval_sample[['review_doi', 'paper_title', 'label']].copy()
results_df_5['prediction'] = preds_5
results_df_5['response']   = resps_5
results_df_5['time_sec']   = [round(t, 2) for t in times_5]

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path_5 = RESULTS_DIR / f'eval_{SAVE_NAME_5}_zero_shot_{ts}.csv'
results_df_5.to_csv(out_path_5, index=False)
print(f'\n✓ Saved to {out_path_5.name}')

Evaluating claude-opus-4-6 (OPTIMISED)  (5,721 samples)
Estimated API cost: ~$123 USD (extended thinking enabled)


claude-opus-4-6 (opt):   0%|          | 0/5721 [00:00<?, ?it/s]


✓ Saved to eval_claude_opus_4_6_optimised_zero_shot_20260307_172954.csv


In [ ]:
# =============================================================================
# Claude Opus 4.6 (Optimised) — Results
# =============================================================================
labels_5 = eval_sample['label'].tolist()
metrics_5 = score(labels_5, preds_5)
metrics_5['time_min'] = round(sum(times_5) / 60, 1)

print(f"\n{'='*60}")
print(f'{MODEL_NAME_5} (OPTIMISED)  RESULTS  ({metrics_5["valid"]}/{len(labels_5)} valid)')
print(f"{'='*60}")
print(f'  Acc  = {metrics_5["acc"]:.4f}')
print(f'  Prec = {metrics_5["prec"]:.4f}')
print(f'  Rec  = {metrics_5["rec"]:.4f}')
print(f'  F1   = {metrics_5["f1"]:.4f}')
print(f'  Time = {metrics_5["time_min"]:.1f} min')

# Confusion matrix
vt_5 = [l for l, p in zip(labels_5, preds_5) if p != -1]
vp_5 = [p for p in preds_5 if p != -1]
if len(vt_5) > 0:
    cm_5 = confusion_matrix(vt_5, vp_5)
    if cm_5.shape == (2, 2):
        tn5, fp5, fn5, tp5 = cm_5.ravel()
        print(f'\n  TP={tp5}  FP={fp5}  FN={fn5}  TN={tn5}')
        print(f'  FNR = {fn5/(tp5+fn5):.1%}  |  FPR = {fp5/(fp5+tn5):.1%}')

# ─── Comparison table ─────────────────────────────────────────────────────
print(f"\n\n{'='*60}")
print("COMPARISON: Claude Opus 4.6 (baseline vs optimised)")
print(f"{'='*60}")
print(f"{'Metric':<10} {'Baseline':>10} {'Optimised':>10} {'Δ':>10}")
print(f"{'-'*42}")
print(f"{'Acc':<10} {metrics_4['acc']:>10.4f} {metrics_5['acc']:>10.4f} {metrics_5['acc']-metrics_4['acc']:>+10.4f}")
print(f"{'Prec':<10} {metrics_4['prec']:>10.4f} {metrics_5['prec']:>10.4f} {metrics_5['prec']-metrics_4['prec']:>+10.4f}")
print(f"{'Rec':<10} {metrics_4['rec']:>10.4f} {metrics_5['rec']:>10.4f} {metrics_5['rec']-metrics_4['rec']:>+10.4f}")
print(f"{'F1':<10} {metrics_4['f1']:>10.4f} {metrics_5['f1']:>10.4f} {metrics_5['f1']-metrics_4['f1']:>+10.4f}")
print(f"{'Time':<10} {metrics_4['time_min']:>9.1f}m {metrics_5['time_min']:>9.1f}m")

print(f"\n{'='*60}")
print('Done.')


claude-opus-4-6 (OPTIMISED)  RESULTS  (5721/5721 valid)
  Acc  = 0.9283
  Prec = 0.8264
  Rec  = 0.6885
  F1   = 0.7512
  Time = 398.4 min

  TP=619  FP=130  FN=280  TN=4692
  FNR = 31.1%  |  FPR = 2.7%


COMPARISON: Claude Opus 4.6 (baseline vs optimised)
Metric       Baseline  Optimised          Δ
------------------------------------------
Acc            0.8832     0.9283    +0.0451
Prec           0.8621     0.8264    -0.0357
Rec            0.3059     0.6885    +0.3826
F1             0.4516     0.7512    +0.2996
Time           159.2m     398.4m

Done.


# Gemini 3.1 Pro Preview — Frontier Reasoning Model (via Google GenAI API)

In [ ]:
# =============================================================================
# Run optimised evaluation — Gemini 3.1 Pro Preview (sensitivity-focused + thinking)
# =============================================================================
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

load_dotenv(project_root / '.env')

MODEL_NAME_6 = 'gemini-3.1-pro-preview'
SAVE_NAME_6  = 'gemini_3_1_pro_preview'

# ─── API client ───────────────────────────────────────────────────────────
gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

# ─── System prompt (same sensitivity-focused approach as Claude optimised) ─
SYSTEM_PROMPT_6 = (
    "You are an expert systematic review screener for Cochrane reviews. "
    "Your task is title/abstract screening — the FIRST pass in a two-stage process. "
    "Papers you INCLUDE will undergo full-text review later, so false inclusions "
    "are acceptable and expected.\n\n"
    "CRITICAL PRINCIPLE: In systematic review screening, SENSITIVITY (recall) is "
    "far more important than specificity. Missing a relevant paper is a serious "
    "error; including a borderline paper is not.\n\n"
    "WHEN IN DOUBT, INCLUDE THE PAPER.\n\n"
    "INCLUDE if the paper MIGHT be relevant — even tangentially — to the review's "
    "scope, population, intervention, or outcomes.\n"
    "EXCLUDE only if the paper is CLEARLY and OBVIOUSLY irrelevant."
)

OPTIMISED_PROMPT_6 = (
    "=== REVIEW SCOPE ===\n"
    "Title: {review_title}\n\n"
    "Abstract: {review_abstract}\n\n"
    "=== CANDIDATE PAPER ===\n"
    "Title: {paper_title}\n\n"
    "Abstract: {paper_abstract}\n\n"
    "=== SCREENING DECISION ===\n"
    "Could this paper potentially be relevant to the review above? "
    "Remember: err on the side of inclusion. Only exclude if clearly irrelevant.\n\n"
    "Respond with exactly one word: INCLUDE or EXCLUDE"
)

def create_prompt_6(row):
    return OPTIMISED_PROMPT_6.format(
        review_title    = str(row['review_title'])[:500],
        review_abstract = str(row['review_abstract'])[:2000],
        paper_title     = str(row['paper_title'])[:300],
        paper_abstract  = str(row['paper_abstract'])[:2000],
    )

# ─── Cost estimation ──────────────────────────────────────────────────────
# ~1,200 input tokens/sample × 5,721 ≈ 6.9M → ~$14
# ~500 output+thinking tokens/sample × 5,721 ≈ 2.9M → ~$35
# Estimated total: ~$49  (at $2/$12 per MTok)
print(f"Evaluating {MODEL_NAME_6} (OPTIMISED)  ({len(eval_sample):,} samples)")
print(f"Estimated API cost: ~$49 USD (thinking enabled)")
print(f"{'='*60}")

preds_6, resps_6, times_6 = [], [], []
for i, (_, row) in enumerate(tqdm(eval_sample.iterrows(), total=len(eval_sample),
                                   desc=MODEL_NAME_6)):
    prompt = create_prompt_6(row)
    try:
        t0 = time.time()
        response = gemini_client.models.generate_content(
            model=MODEL_NAME_6,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT_6,
                thinking_config=types.ThinkingConfig(thinking_budget=8192),
                temperature=0.2,
                max_output_tokens=256,
            ),
        )
        resp_text = response.text or ''
        elapsed = time.time() - t0
        pred = extract_decision(resp_text)
    except Exception as e:
        err_msg = str(e)
        if '429' in err_msg or 'RESOURCE_EXHAUSTED' in err_msg:
            time.sleep(30)
            try:
                t0 = time.time()
                response = gemini_client.models.generate_content(
                    model=MODEL_NAME_6,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT_6,
                        thinking_config=types.ThinkingConfig(thinking_budget=8192),
                        temperature=0.2,
                        max_output_tokens=256,
                    ),
                )
                resp_text = response.text or ''
                elapsed = time.time() - t0
                pred = extract_decision(resp_text)
            except Exception as e2:
                resp_text, pred, elapsed = str(e2), -1, 0
        else:
            resp_text, pred, elapsed = err_msg, -1, 0

    preds_6.append(pred)
    resps_6.append(resp_text)
    times_6.append(elapsed)

# Build & save results
results_df_6 = eval_sample[['review_doi', 'paper_title', 'label']].copy()
results_df_6['prediction'] = preds_6
results_df_6['response']   = resps_6
results_df_6['time_sec']   = [round(t, 2) for t in times_6]

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path_6 = RESULTS_DIR / f'eval_{SAVE_NAME_6}_zero_shot_{ts}.csv'
results_df_6.to_csv(out_path_6, index=False)
print(f'\n✓ Saved to {out_path_6.name}')

Evaluating gemini-3.1-pro-preview (OPTIMISED)  (5,721 samples)
Estimated API cost: ~$49 USD (thinking enabled)


gemini-3.1-pro-preview:   0%|          | 0/5721 [00:00<?, ?it/s]

In [ ]:
# =============================================================================
# Gemini 3.1 Pro Preview — Results
# =============================================================================
labels_6 = eval_sample['label'].tolist()
metrics_6 = score(labels_6, preds_6)
metrics_6['time_min'] = round(sum(times_6) / 60, 1)

print(f"\n{'='*60}")
print(f'{MODEL_NAME_6}  RESULTS  ({metrics_6["valid"]}/{len(labels_6)} valid)')
print(f"{'='*60}")
print(f'  Acc  = {metrics_6["acc"]:.4f}')
print(f'  Prec = {metrics_6["prec"]:.4f}')
print(f'  Rec  = {metrics_6["rec"]:.4f}')
print(f'  F1   = {metrics_6["f1"]:.4f}')
print(f'  Time = {metrics_6["time_min"]:.1f} min')

# Confusion matrix
vt_6 = [l for l, p in zip(labels_6, preds_6) if p != -1]
vp_6 = [p for p in preds_6 if p != -1]
if len(vt_6) > 0:
    cm_6 = confusion_matrix(vt_6, vp_6)
    if cm_6.shape == (2, 2):
        tn6, fp6, fn6, tp6 = cm_6.ravel()
        print(f'\n  TP={tp6}  FP={fp6}  FN={fn6}  TN={tn6}')
        print(f'  FNR = {fn6/(tp6+fn6):.1%}  |  FPR = {fp6/(fp6+tn6):.1%}')

# ─── Full comparison table ────────────────────────────────────────────────
print(f"\n\n{'='*60}")
print("FULL COMPARISON — All Models (optimised prompt where applicable)")
print(f"{'='*60}")
print(f"{'Model':<35} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'Time':>8}")
print(f"{'-'*73}")
print(f"{'claude-opus-4-6 (opt)':<35} {metrics_5['acc']:>7.4f} {metrics_5['prec']:>7.4f} {metrics_5['rec']:>7.4f} {metrics_5['f1']:>7.4f} {metrics_5['time_min']:>7.1f}m")
print(f"{'gemini-3.1-pro-preview':<35} {metrics_6['acc']:>7.4f} {metrics_6['prec']:>7.4f} {metrics_6['rec']:>7.4f} {metrics_6['f1']:>7.4f} {metrics_6['time_min']:>7.1f}m")

print(f"\n{'='*60}")
print('Done.')